## 📌 1. Title & Intro

In [ ]:
# =========================================
# 🧠 TensorFlow ML Pipeline - End-to-End
# Author: Ra'uf
# =========================================

"""
This notebook demonstrates a complete machine learning workflow:
- Data loading
- Preprocessing
- Model training
- Evaluation
"""

## 📦 2. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

## 📥 3. Load Dataset

In [ ]:
DATA_PATH = "../data/raw/data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

## 🔍 4. Basic EDA (Exploratory Data Analysis)

In [ ]:
print("Shape:", df.shape)
print("\nInfo:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

df.describe()

## 🧹 5. Preprocessing (Simple Version for Notebook)

In [ ]:
TARGET = "target"

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Handle numeric only (simple demo)
X = X.select_dtypes(include=np.number)

# Fill missing
X = X.fillna(X.median())

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## 🧠 6. Build Model (TensorFlow)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 🚀 7. Training

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=1
)

## 📈 8. Training Visualization

In [ ]:
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.show()

## 🎯 9. Evaluation

In [ ]:
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))

## 📊 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 📉 11. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1],[0,1],"--")
plt.legend()
plt.title("ROC Curve")
plt.show()

## 💾 12. Save Model

In [ ]:
model.save("../models/notebook_model")